# 37 — Bullet Parsing & STAR Scoring
**Goal:** Score resume bullets on action verbs, metrics, and STAR compliance.

## 1. STAR Method Explained

In [ ]:
print('''STAR = Situation, Task, Action, Result
Good bullet (STAR compliant):
  "Reduced model latency by 40% by optimizing inference pipeline"
  Action: Reduced | Result: 40% | Task: optimizing inference

Weak bullet (no STAR):
  "Was responsible for ML models"
  → Passive voice, no metric, no result

Scoring criteria:
  ✓ Strong action verb (Developed, Led, Reduced, Built)
  ✓ Quantified metric (% improvement, $, count)
  ✓ Specific context (technology, team size)
  ✓ Result/outcome mentioned''')

## 2. Bullet Scorer

In [ ]:
import re

ACTION_VERBS = {"developed", "led", "reduced", "built", "designed", "implemented",
    "created", "managed", "delivered", "achieved", "improved", "increased",
    "decreased", "optimized", "architected", "spearheaded", "established",
    "launched", "generated", "transformed", "engineered"}

def score_bullet(bullet):
    """Score a single resume bullet 0-1."""
    text = bullet.strip()
    score = 0
    
    # Check starts with action verb
    first_word = text.split()[0].lower().rstrip(",.;:")
    if first_word in ACTION_VERBS:
        score += 0.3
    
    # Check for quantified metrics
    if re.search(r"\d+\s*(%|million|billion|\$|x|percent)", text, re.IGNORECASE):
        score += 0.3
    if re.search(r"\bby\s+\d+", text, re.IGNORECASE):
        score += 0.15
    
    # Check for specific context
    if re.search(r"(using|with|via)\s+[A-Z]", text):
        score += 0.15
    if any(word in text.lower() for word in ["team", "pipeline", "system", "platform"]):
        score += 0.1
    
    return min(score, 1.0)

bullets = [
    "Reduced model latency by 40% through TensorFlow optimization",
    "Was responsible for ML model development",
    "Led team of 5 engineers to deliver ML platform",
    "Improved customer engagement with data-driven recommendations",
    "Worked on various projects",
]

for b in bullets:
    s = score_bullet(b)
    print(f"  {'WEAK' if s < 0.4 else 'GOOD' if s < 0.7 else 'STRONG'} ({s:.2f}) '{b[:50]}'")

## 3. STAR Compliance Check

In [ ]:
def star_compliance(bullet):
    checks = {
        "has_action_verb": bool(re.search(r"^(" + "|".join(ACTION_VERBS) + r")", bullet, re.IGNORECASE)),
        "has_metric": bool(re.search(r"\d+", bullet)),
        "has_technology": bool(re.search(r"[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*", bullet)),
        "has_outcome": bool(re.search(r"(by|resulting|achieving|increasing|reducing)", bullet, re.IGNORECASE)),
        "is_active_voice": not bool(re.search(r"^was|^were|^had|^has been", bullet, re.IGNORECASE)),
    }
    return checks

for b in bullets:
    c = star_compliance(b)
    passed = sum(1 for v in c.values() if v)
    print(f"  {passed}/5 STAR: '{b[:45]}'")
    for check, result in c.items():
        print(f"    {check:20s}: {'✓' if result else ' '}")

## Summary: Bullet scoring identifies weak bullets for improvement. Rule-based, no LLM needed.